# External-gated versus plain-self-trigger comparison

This notebook compares the two acquisition schemes using trigger timestamp continuity, not wall-time trigger estimates. The expected decoded trigger interval is $\Delta t_0=10^9/(3r)$ ns. For each adjacent timestamp gap, $k=\mathrm{round}(\Delta t/\Delta t_0)$ and the gap contributes $k-1$ missing trigger slots when $|\Delta t/\Delta t_0-k|\leq0.30$.

Trigger retention is $N_{\mathrm{observed}}/N_{\mathrm{inferred\ slots}}$, so it cannot exceed 100%. This detects internal gaps only; timestamp coverage is shown separately because missing records before the first or after the last received record cannot be inferred.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/tools/sampic_tests').is_dir())
SCAN_ROOT = REPO / 'scripts/tools/sampic_tests/data/external_trigger_batching_scan/pipelined_short_test/20260824_220030'
EXPECTED_CHANNELS = set(range(25, 32))
TIMESTAMP_SCALE_FACTOR = 3.0
GAP_TOLERANCE = 0.30
TIMESTAMP_MODULUS_NS = float(2**40) * 10.0
WAVE_GAP_NS = 100.0
LABEL = {'l2_external_gate': 'L2 + external gate', 'self_trigger': 'Plain self-trigger'}
COLOR = {'l2_external_gate': '#4c78a8', 'self_trigger': '#e45756'}
SCAN_ROOT

In [ ]:
def trigger_continuity(triggers, rate_hz):
    ordered = triggers.sort_values(['event_index', 'trigger_index'])
    raw = ordered['timestamp_ns'].to_numpy(float)
    if len(raw):
        wrap = np.r_[0, np.cumsum(np.diff(raw) < -0.5 * TIMESTAMP_MODULUS_NS)]
        timestamp = raw + wrap * TIMESTAMP_MODULUS_NS
    else:
        timestamp = raw
    dt0 = 1e9 / (TIMESTAMP_SCALE_FACTOR * rate_hz)
    ratio = np.diff(timestamp) / dt0
    nearest = np.maximum(1, np.rint(ratio).astype(np.int64))
    compatible = np.abs(ratio - nearest) <= GAP_TOLERANCE
    steps = np.where(compatible, nearest, 1)
    slots = int(len(timestamp)) if len(timestamp) < 2 else int(1 + steps.sum())
    return {
        'ratios': ratio,
        'observed': len(timestamp),
        'slots': slots,
        'missing': int(np.maximum(0, steps - 1).sum()),
        'retention_percent': 100 * len(timestamp) / slots if slots else np.nan,
        'span_s': (timestamp[-1] - timestamp[0]) / 1e9 if len(timestamp) > 1 else 0.0,
        'irregular': int((~compatible).sum()),
        'rollovers': int(np.sum(np.diff(raw) < -0.5 * TIMESTAMP_MODULUS_NS)),
    }

def hit_wave_counts(expected_hits):
    ordered = expected_hits.sort_values('timestamp_ns').copy()
    if ordered.empty:
        return 0, 0
    t = ordered['timestamp_ns'].to_numpy(float)
    ordered['wave'] = np.r_[0, np.cumsum(np.diff(t) > WAVE_GAP_NS)]
    channels = ordered.groupby('wave')['channel'].nunique()
    return len(channels), int(channels.ge(7).sum())

In [ ]:
rows, gap_ratios = [], {}
for run_dir in sorted(SCAN_ROOT.glob('rate_*')):
    metadata = json.loads((run_dir / 'metadata.json').read_text())
    acquisition = metadata['acquisition']
    scheme = acquisition['acquisition_scheme']
    rate_hz = float(acquisition['lecroy_readback_rate_hz'])
    triggers = pd.read_csv(run_dir / 'triggers.csv')
    hits = pd.read_csv(run_dir / 'hits.csv', usecols=['feb', 'sampic', 'channel', 'timestamp_ns', 'accepted', 'nearest_trigger_sorted_index'])
    expected = hits[hits['feb'].eq(1) & hits['sampic'].eq(1) & hits['channel'].isin(EXPECTED_CHANNELS)].copy()
    matched = expected[expected['accepted'].eq(1)]
    complete_triggers = int(matched.groupby('nearest_trigger_sorted_index')['channel'].nunique().ge(7).sum())
    continuity = trigger_continuity(triggers, rate_hz)
    waves, complete_waves = hit_wave_counts(expected)
    gap_ratios[(scheme, rate_hz)] = continuity['ratios']
    rows.append({
        'scheme': scheme, 'rate_hz': rate_hz, 'trigger_records': len(triggers),
        'inferred_trigger_slots': continuity['slots'], 'inferred_missing': continuity['missing'],
        'trigger_retention_percent': continuity['retention_percent'],
        'trigger_timestamp_span_s': continuity['span_s'], 'irregular_gaps': continuity['irregular'],
        'timestamp_rollovers': continuity['rollovers'], 'expected_channel_hits': len(expected),
        'matched_expected_channel_hits': len(matched),
        'expected_hit_match_percent': 100 * len(matched) / max(1, len(expected)),
        'complete_received_triggers': complete_triggers,
        'complete_given_received_percent': 100 * complete_triggers / max(1, len(triggers)),
        'complete_per_inferred_slot_percent': 100 * complete_triggers / max(1, continuity['slots']),
        'observed_hit_waves': waves, 'complete_hit_waves': complete_waves,
        'wave_completeness_percent': 100 * complete_waves / max(1, waves),
        'waves_per_inferred_slot': waves / max(1, continuity['slots']),
    })

summary = pd.DataFrame(rows).sort_values(['scheme', 'rate_hz']).reset_index(drop=True)
healthy_span = summary['trigger_timestamp_span_s'].median()
summary['short_trigger_coverage'] = summary['trigger_timestamp_span_s'] < 0.10 * healthy_span
summary

## Trigger continuity and seven-hit completeness

A black cross marks a capture with less than 10% of the median trigger timestamp span. Such a point is retained for diagnosis but should not be interpreted as an ordinary full-span rate measurement. Conditional completeness uses only received trigger records; continuity-adjusted completeness includes timestamp-inferred missing trigger slots in the denominator.

In [ ]:
metrics = [
    ('trigger_retention_percent', 'Observed trigger records / inferred slots (%)'),
    ('trigger_timestamp_span_s', 'First-to-last trigger timestamp span (s)'),
    ('complete_given_received_percent', 'Received triggers containing all 7 channels (%)'),
    ('complete_per_inferred_slot_percent', 'Inferred trigger slots containing all 7 channels (%)'),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (column, title) in zip(axes.flat, metrics):
    for scheme, group in summary.groupby('scheme'):
        group = group.sort_values('rate_hz')
        ax.plot(group['rate_hz'] / 1000, group[column], 'o-', color=COLOR[scheme], label=LABEL[scheme])
        short = group[group['short_trigger_coverage']]
        ax.scatter(short['rate_hz'] / 1000, short[column], marker='x', s=120, linewidth=2, color='black', zorder=5)
    ax.set(title=title, xlabel='Lecroy frequency readback (kHz)')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
for ax in (axes[0, 0], axes[1, 0], axes[1, 1]):
    ax.yaxis.set_major_formatter(PercentFormatter(100))
fig.tight_layout()

## Complete hit waves independent of trigger association

Expected-channel hits are grouped into waves when adjacent timestamps are no more than 100 ns apart. This checks whether the seven channels arrived together even when a wave did not associate with an external-trigger record. Plain self-trigger mode can produce multiple complete waves per external-counter opportunity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
wave_metrics = [
    ('wave_completeness_percent', 'Observed hit waves containing all 7 channels (%)'),
    ('waves_per_inferred_slot', 'Observed hit waves / inferred trigger slot'),
    ('expected_hit_match_percent', 'Expected-channel hits matched to a trigger (%)'),
]
for ax, (column, title) in zip(axes, wave_metrics):
    for scheme, group in summary.groupby('scheme'):
        group = group.sort_values('rate_hz')
        ax.plot(group['rate_hz'] / 1000, group[column], 'o-', color=COLOR[scheme], label=LABEL[scheme])
    ax.set(title=title, xlabel='Lecroy frequency readback (kHz)')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
axes[0].yaxis.set_major_formatter(PercentFormatter(100))
axes[2].yaxis.set_major_formatter(PercentFormatter(100))
fig.tight_layout()

## Adjacent trigger gaps

The green band is the accepted one-period interval, $1\pm0.30$. Peaks near integer values above one indicate internally missing trigger records. The 20 kHz plain-self-trigger capture contains one much larger integer-compatible gap; the displayed histogram clips very large ratios at its right edge while the inferred count uses the full value.

In [ ]:
rates = sorted(summary['rate_hz'].unique())
fig, axes = plt.subplots(len(rates), 2, figsize=(12, 12))
for row, rate_hz in enumerate(rates):
    for column, scheme in enumerate(['l2_external_gate', 'self_trigger']):
        ax = axes[row, column]
        ratio = gap_ratios[(scheme, rate_hz)]
        bins = np.arange(0.5, 10.6, 1.0)
        ax.hist(np.clip(ratio, bins[0], bins[-1] - 1e-9), bins=bins, color=COLOR[scheme], alpha=0.8)
        ax.axvspan(1 - GAP_TOLERANCE, 1 + GAP_TOLERANCE, color='#54a24b', alpha=0.16)
        ax.set_yscale('log')
        ax.set(title=f'{LABEL[scheme]}, {rate_hz/1000:g} kHz', xlabel=r'$\Delta t/\Delta t_0$', ylabel='Adjacent intervals')
        ax.grid(alpha=0.2)
fig.tight_layout()